# Capstone — AI Referral Opportunity Ranking
### Mirrors the deployed research paper (see `submission/paper_url.txt`)

**Lane:** AI Referral Opportunity (freestyle) · **Data:** FlyRank ML Internship warehouse, `month=2026-03` partition · **Author:** Hussain Afroz Khan


This notebook is the working copy behind the deployed paper. It re-states, in order, what
`w01`-`w07` already built and validated -- it does not re-derive new findings. Every number
below traces back to a specific earlier notebook, named in each section.

## 1. Question

**Research question:** Among content pages with real, current Google-search demand, which ones
most resemble pages that already receive AI-referred sessions -- so a content team knows where to
look first for AI-visibility opportunity?

- **Unit of analysis:** one content item (page), summarized over `month=2026-03`.
- **Decision this supports:** if a content/SEO team can only review a handful of pages this
  month, which ones are worth reviewing first for AI-referral opportunity?
- **Why not classification:** `has_ai_sessions_month` is sparse -- 30,177 of 78.8M warehouse rows
  warehouse-wide, ~8.6% of the demand-worthy monthly slice used here. That base rate is too thin
  to trust a binary classifier's AUC/accuracy; the honest task is **ranking**, validated with
  **lift@K** against the observed base rate. (Full reasoning: `w02_ml_task_framing.ipynb`, ML-03.)
- **Cost of a wrong call:** a false positive costs a few minutes of an analyst's review time; a
  false negative -- a real opportunity that never enters the queue -- costs nothing *visible*,
  which is the more dangerous failure. This shapes the review rules in Section 6 below.

In [1]:
print("Section 1 sources: w01_research_question.ipynb, w02_ml_task_framing.ipynb")

Section 1 sources: w01_research_question.ipynb, w02_ml_task_framing.ipynb


## 2. Data

- **Release:** FlyRank ML Internship Hugging Face warehouse (`hf://datasets/FlyRank/internship-warehouse`),
  queried with DuckDB. Never the sealed `_sample` table.
- **Tables:** `fact_content_daily_performance` (daily grain, joined up to one row per content
  item for the month) + `dim_content` (static content metadata).
- **Partition:** `month=2026-03` only -- a mid-panel development month, per the internship's
  iteration rule. 9,841,378 daily rows in this partition; 331,437 distinct content items;
  55 clients.
- **Filters applied, and why:**
  - `ga4_data_available IS TRUE` (three-valued flag -- never filtered with `= FALSE` / `NOT`,
    which silently mishandles NULLs) -> 90,489 content items.
  - Demand-worthy floor, `total_gsc_impressions_month >= 100` -> 32,596 rows, 30 clients,
    used as the **labeled population** for fitting/validating the model (base rate of
    `has_ai_sessions_month`: 8.61%).
  - The **delivered opportunity queue** further restricts to zero-AI-session rows within the
    GA4-available population: 29,788 of 90,489 content items (32.9%).
- **Excluded, and why (public-safe):** any GA4 "total sessions" aggregate that isn't already
  split from AI-referred sessions (near-circular with the label); any FlyRank product score
  (`priority_score`, `health_score`); no client names, URLs, or raw queries appear anywhere in
  this repo -- `client_hash_id` / `content_hash_id` are pseudonymous grouping keys only, never
  features. Full contract: `w03_data_contract.ipynb`.

In [2]:
print("Data contract verified in w03_data_contract.ipynb: 0 duplicate-grain rows, "
      "9,841,378 rows in month=2026-03, 90,489 GA4-available content items.")

Data contract verified in w03_data_contract.ipynb: 0 duplicate-grain rows, 9,841,378 rows in month=2026-03, 90,489 GA4-available content items.


## 3. Methodology

**Features (five, plus two carried in from the signal audit):**
`total_gsc_impressions_month`, `avg_gsc_position_month`, `days_with_impressions_month`,
`word_count`, `content_type` -- plus `days_since_update_month` (kept in the model despite a
MIXED signal verdict, see below) and missingness flags (`has_word_count`, etc.).

**Label / proxy:** `has_ai_sessions_month` (`SUM(sessions_ai) > 0` within the month) -- an
**evidence variable**, never a forecast target. It is read from GA4 session data in the *same*
month as the features, so this work ranks by current resemblance, not future prediction.

**Signal checks (`w04_baseline_score.ipynb`, ML-07) -- run before building anything:**

| Signal | Verdict | Evidence |
|---|---|---|
| Demand volume (impressions) | **CONFIRMED** | Monotonic climb across impression tiers: 1.7% -> 3.0% -> 7.3% -> 19.1% AI-session rate (base rate 4.19%) |
| Staleness (days since update) | **MIXED** | Non-monotonic buckets (3.9% -> 5.7% -> 1.7%), weak positive correlation (+0.011) -- the opposite of the "stale pages underperform" intuition. Kept out of the baseline rule on purpose. |

**Baseline (`w04_baseline_score.ipynb`):** a transparent rule -- rank the zero-AI-session,
demand-worthy pool by impression-volume percentile alone. No fitting, one reason code
(`high_demand_zero_ai_sessions`), one action (`review_for_ai_visibility`).

**Model (`w05_model.ipynb`, ML-08):** Logistic Regression as the primary model -- interpretable,
extends the baseline's transparent-rule philosophy to several combined signals instead of one.
Random Forest as a secondary check. Gradient Boosting was excluded on purpose (complexity
discipline -- the lane doesn't need it, and the label is too sparse to reward the extra variance).

**Validation design:** `GroupShuffleSplit` by `client_hash_id` (never a random row split) --
content items from the same client share hidden character (template, SEO maturity, which AI
tools happen to crawl that domain), so a random split would let a model partially memorize
"which client is this." 75% of *clients* to train, 25% held out entirely.

**Leakage checks (`w06_validation_audit.ipynb`, ML-09):**
- Random split vs grouped split, same pipeline: the random split showed **26 of ~30 clients
  leaking across train/test**; the grouped split showed **0**.
- `sessions_ai_month` (the raw count the label is thresholded from) was deliberately re-added
  as a feature: lift@K jumped from an honest 5.02x to **19.97x** -- the signature of a label
  leak, not a real result. It stays excluded from every model above.

In [3]:
print("Methodology verified in w04_baseline_score.ipynb (ML-07), w05_model.ipynb (ML-08), "
      "w06_validation_audit.ipynb (ML-09).")

Methodology verified in w04_baseline_score.ipynb (ML-07), w05_model.ipynb (ML-08), w06_validation_audit.ipynb (ML-09).


## 4. Results (vs baseline)

All three scores below are evaluated on the **same held-out test set** (clients never seen in
training), at the same K, against the same base rate -- `w05_model.ipynb`, Section 3.

| Model | K | Base rate | Precision@K | Lift@K |
|---|---|---|---|---|
| Baseline (impression-rank rule, ML-07) | 244 | 3.67% | 11.07% | **3.01x** |
| Logistic Regression | 244 | 3.67% | 18.44% | **5.02x** |
| Random Forest | 244 | 3.67% | 13.93% | **3.79x** |

**Reading:** Logistic Regression beats the transparent baseline by combining the same
non-leaky signals (impressions, position, days-with-impressions, word count, content type,
freshness) into one score -- a ~5x lift over chance, versus the baseline's ~3x. Random Forest
also beats the baseline but trails Logistic Regression here, consistent with a small, sparse
positive class rewarding a simpler, more regularized model.

**What the model leans on (interpretation, ML-08 sec.4):** `days_with_impressions_month` and
`content_type` (feedly-article pages score higher than keyword-article pages) rank highest by
both LR coefficient magnitude and Random Forest permutation importance -- an independent,
non-leaky Search Console/content-metadata signal, not a "too good to be true" one. Freshness
(`days_since_update_month`) ranks low in both models, consistent with its MIXED signal-audit
verdict -- the model, given the chance to use it anyway, still doesn't lean on it much.

In [4]:
# Recreate the results table above (paste real DataFrame code once re-run with HF_TOKEN)
import pandas as pd
results = pd.DataFrame([
    {"model": "Baseline (impression rank, ML-07)", "K": 244, "base_rate": "3.67%", "precision_at_K": "11.07%", "lift_at_K": "3.01x"},
    {"model": "Logistic Regression",               "K": 244, "base_rate": "3.67%", "precision_at_K": "18.44%", "lift_at_K": "5.02x"},
    {"model": "Random Forest",                     "K": 244, "base_rate": "3.67%", "precision_at_K": "13.93%", "lift_at_K": "3.79x"},
])
results

,model,K,base_rate,precision_at_K,lift_at_K
0,"Baseline (impression rank, ML-07)",244,3.67%,11.07%,3.01x
1,Logistic Regression,244,3.67%,18.44%,5.02x
2,Random Forest,244,3.67%,13.93%,3.79x


## 5. Limitations

- **A same-month evidence variable, not a forecast.** `has_ai_sessions_month` is read from the
  *same* month as the features. This work ranks pages by how closely they resemble pages that
  already show AI-referred sessions this month -- it does not predict *future* AI-referral
  traffic, and it does not diagnose *why* a page lacks AI-referred sessions.
- **Single snapshot.** One warehouse partition (`month=2026-03`). No trend, no time-aware
  validation -- a fresh partition would be needed to confirm the ranking holds over time.
- **Sparse positives, small counts.** ~8.6% base rate in the labeled slice; lift@K numbers are
  directional evidence, not statistically robust point estimates, and should not be over-read
  to the third decimal (library-version note: Random Forest results shift slightly across
  scikit-learn/numpy versions -- the stable claim is the multiple-x lift, not the exact figure).
- **Unbalanced client panel.** Per-client tracking history and volume differ widely -- a
  `has_ai_sessions_month = 0` reading can reflect thin GA4 tracking for that client rather than
  a genuine absence of opportunity. Filtering on `ga4_data_available IS TRUE` reduces, but does
  not remove, this confound.
- **Decision-support only.** This queue tells a team where to look first. It is not proof that
  editing any single page will produce AI referral, and it never claims to predict how any AI
  assistant's retrieval works.

In [6]:
print("No code needed ")

No code needed 


## 6. Ranked recommendations

The validated Logistic Regression (refit on the full labeled population, per standard
deployment practice once generalization was confirmed in Section 4) scores the delivered
opportunity queue: 29,788 demand-worthy, zero-AI-session content items across 30 clients.

**Archetype mix** (`w07_action_playbook.ipynb`, ML-10):

| Archetype | Share | What it means |
|---|---|---|
| `page1_zero_ai_referral` | 59.3% | Already ranking well in Google search, zero observed AI-referred sessions |
| `low_affinity_content_type` | 40.1% | Content type historically under-associated with AI referral in this fit |
| `page1_high_affinity_gap` | 0.4% | Page-1 Google ranking + high-affinity content type, still zero AI sessions -- highest-confidence review candidates |
| `underperforming_high_affinity_type` | 0.1% | High-affinity type, weak search performance -- content gap, not a visibility gap |
| `general_opportunity` | 0.0% | Everything else |

**Confidence tiers:** high -- 648 rows, medium -- 1,682 rows, low -- 27,458 rows.

**The playbook, in order:**
1. **Start with `page1_high_affinity_gap`, high confidence** -- smallest group, cheapest to
   trust, most page-ready.
2. **Then `page1_zero_ai_referral`, medium/high confidence** -- already earning search demand;
   review structure, headings, and FAQ coverage for AI-assistant readability.
3. **Treat `low_affinity_content_type` and every `low`-confidence row as a research queue, not
   an action queue** -- 92.2% of the delivered rows carry a mandatory human-review flag before
   any content change, weighted deliberately toward low-confidence and thin-client rows (a false
   negative here is invisible and costly; a false positive just costs a few minutes of review).
4. **A human must confirm, before acting on any row:** the client's GA4 property really shows
   zero AI-referred sessions (not a tracking gap), and the page's content type/intent genuinely
   matches the archetype assigned.

**Monitoring:** re-validate if a fresh grouped-split lift@K falls notably below ~5x or
approaches the baseline's ~3x, or if a new partition's base rate moves >2x from this month's
8.61% reading (`w07_action_playbook.ipynb`, sec.4).

In [7]:
print("Ranked recommendations sourced from w07_action_playbook.ipynb (ML-10).")


Ranked recommendations sourced from w07_action_playbook.ipynb (ML-10).


## 7. Artifacts the paper embeds

The deployed paper (`docs/index.html`) embeds three charts built directly from the numbers
above -- no chart requires re-downloading data to reproduce, since every number traces to a
notebook cited next to it:

1. **Lift@K comparison** -- baseline vs. Logistic Regression vs. Random Forest (Section 4).
2. **Volume-signal bucket chart** -- AI-session rate by impression tier (Section 3, signal
   checks).
3. **Archetype mix** -- the delivered opportunity queue's composition (Section 6).

If you want to regenerate these as image files instead of inline SVG: re-run
`w07_action_playbook.ipynb` Section "Exports for the paper" -- it already writes
`work/figures/w07_archetype_mix.svg` and `work/figures/w07_confidence_mix.svg`, and a small
`matplotlib` bar chart from the Section 4 table above can be saved the same way.

In [9]:
print("Run this cell in Colab (with the real results DataFrame) to export a lift@K chart "
      "as a committed SVG, matching the pattern already used in w07_action_playbook.ipynb.")

Run this cell in Colab (with the real results DataFrame) to export a lift@K chart as a committed SVG, matching the pattern already used in w07_action_playbook.ipynb.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.